# RDD Optimised — Naive Bayes Classifier

RDD implementation with four targeted optimisations over the baseline.
All logic is identical to `naive_bayes_rdd_baseline.ipynb` except for these four changes:

| # | Optimisation | Baseline | Optimised |
|---|---|---|---|
| 1 | Map phase | `flatMap` — one `(key, 1)` per row | `mapPartitions` — local pre-aggregation per partition |
| 2 | Reduce phase | `groupByKey` — full shuffle | `reduceByKey` — partial reduce before shuffle |
| 3 | Probability table | Serialised per task | `sc.broadcast()` — cached per executor |
| 4 | Count RDD | Recomputed on each action | `.persist()` — materialised after first action |

## Pipeline
| Cell | Role |
|------|------|
| 1 | SparkSession, imports, path setup |
| 2 | Load data → `(train_rdd, test_rdd)` of `(label, [features])` |
| 3 | MapReduce training with optimisations 1, 2, 4 → `all_counts` dict |
| 4 | `compute_log_probs()` → `log_prob_table` |
| 5 | Broadcast (opt 3) + predict on test set → `predictions_list` |
| 6 | Evaluate accuracy, print timings |

`PYTHONPATH` is set before `SparkSession` creation so worker subprocesses inherit it
and can import from `core/`. `FEATURE_COLS` and `LABEL_COL` are not imported — this
notebook accesses features by position via the `(label, [features])` tuple format,
never by column name.

Reference: Zheng, S. (2014). *Naïve Bayes Classifier: A MapReduce Approach*. NDSU.

In [ ]:
import time
import math
import sys
import os

from pyspark.sql import SparkSession

project_root = os.path.abspath('..')
sys.path.insert(0, project_root)
# Set before SparkSession so worker subprocesses inherit it and can import from core/.
os.environ['PYTHONPATH'] = f"{project_root}:{os.environ.get('PYTHONPATH', '')}".strip(':')

from core.naive_bayes import compute_log_probs, predict, evaluate
from data.loader import load_car_rdd

# TODO (Databricks): Remove .master() — the cluster provides its own master URL.
spark = (
    SparkSession.builder
    .master('local[*]')
    .appName('NaiveBayes-RDD-Optimised')
    .getOrCreate()
)
sc = spark.sparkContext
sc.setLogLevel('WARN')
print('SparkSession ready.')

In [ ]:
# Cell 2 — Load data
#
# TODO: Replace None with your dataset path.
#   Local      : '/Users/you/data/car.data'
#   Databricks : 'dbfs:/FileStore/car.data'
DATA_PATH = None

train_rdd, test_rdd = load_car_rdd(spark, filepath=DATA_PATH)

train_count = train_rdd.count()
test_count  = test_rdd.count()
print(f'Train rows : {train_count}')
print(f'Test rows  : {test_count}')
print(f'Sample row : {train_rdd.first()}')

class_dist = train_rdd.map(lambda row: row[0]).countByValue()
print(f'Class distribution: {dict(class_dist)}')

In [ ]:
# Cell 3 — Training phase (MapReduce)
#
# OPTIMISATION 1: mapPartitions instead of flatMap
# flatMap emits one (key, 1) pair per feature per row — O(rows * features) pairs shuffled.
# mapPartitions processes an entire partition at once, accumulating counts locally
# before emitting. Each partition emits at most (classes * features * unique_values)
# pairs — far fewer than (rows * features) when the dataset is large.
# This dramatically reduces the volume of data sent across the network.

t_train_start = time.time()

def map_partition(rows):
    # Accumulate all counts for this partition into a local Python dict.
    # No data leaves this function until yield — all combining is in memory.
    local_counts = {}
    for label, features in rows:
        for feat_idx, value in enumerate(features):
            key = f'feat_{feat_idx}_{value}_{label}'
            local_counts[key] = local_counts.get(key, 0) + 1
        class_key = f'class_{label}'
        local_counts[class_key] = local_counts.get(class_key, 0) + 1
    # Yield only the aggregated (key, partial_count) pairs — much less data than 1s.
    # Input  (per partition): many rows of (label, [features])
    # Output (per partition): ('feat_0_low_unacc', 42), ('class_unacc', 80), ...
    yield from local_counts.items()

# mapPartitions calls map_partition once per partition, not once per row.
# Input:  RDD partition of (label, [features])
# Output: RDD of (key, partial_count) already aggregated within each partition
mapped_rdd = train_rdd.mapPartitions(map_partition)

# OPTIMISATION 2: reduceByKey instead of groupByKey
# groupByKey ships ALL (key, value) pairs to a single node before summing.
# reduceByKey applies the lambda LOCALLY on each worker first (combining partial
# counts within the same partition), then shuffles only the partial sums.
# Result: far less data crosses the network for the same final output.
# Input:  ('feat_0_low_unacc', 42) on worker A, ('feat_0_low_unacc', 58) on worker B
# Output: ('feat_0_low_unacc', 100)  — after combining on one node
counts_rdd = mapped_rdd.reduceByKey(lambda a, b: a + b)

# OPTIMISATION 4: persist the count RDD
# Spark uses lazy evaluation: every action triggers a full recomputation from source.
# If counts_rdd were used in more than one action (e.g. count() then collect()),
# the entire mapPartitions + shuffle would run twice. persist() materialises the RDD
# in memory after the first action so subsequent actions reuse the cached result.
counts_rdd.persist()

all_counts = dict(counts_rdd.collect())

train_time = time.time() - t_train_start
print(f'[TIMING] Training: {train_time:.4f}s')
print(f'Total unique keys : {len(all_counts)}')
print(f'Sample counts     : {list(all_counts.items())[:4]}')

In [ ]:
# Separate all_counts into the two input dicts compute_log_probs() expects.
class_counts   = {k[6:]: v for k, v in all_counts.items() if k.startswith('class_')}
feature_counts = {k: v       for k, v in all_counts.items() if k.startswith('feat_')}
class_totals   = class_counts  # same dict for standard Naive Bayes

log_prob_table = compute_log_probs(class_counts, feature_counts, class_totals)

# Log-probabilities should all be negative (all P < 1).
print(f"Classes           : {log_prob_table['classes']}")
print(f"Log class probs   : {log_prob_table['log_class_probs']}")
print(f"Sample feat probs : {list(log_prob_table['log_feature_probs'].items())[:3]}")

In [ ]:
# OPTIMISATION 3: broadcast sends log_prob_table once per executor instead of
# serialising a full copy with every task.
bc_log_prob_table = sc.broadcast(log_prob_table)

t_pred_start = time.time()

# Input row:  ('unacc', ['low', 'med', '2', '2', 'small', 'low'])
# Output   :  ('unacc', 'unacc')  <- (true_label, predicted_label)
predictions_rdd = test_rdd.map(
    lambda row: (row[0], predict(bc_log_prob_table.value, row[1]))
)
predictions_list = predictions_rdd.collect()

pred_time = time.time() - t_pred_start
print(f'[TIMING] Prediction: {pred_time:.4f}s')
print(f'Sample predictions : {predictions_list[:5]}')

In [ ]:
# Cell 6 — Evaluation

true_labels = [pair[0] for pair in predictions_list]
pred_labels = [pair[1] for pair in predictions_list]

accuracy = evaluate(pred_labels, true_labels)

print(f'Accuracy     : {accuracy:.4f} ({accuracy * 100:.2f}%)')
print(f'Train time   : {train_time:.4f}s')
print(f'Predict time : {pred_time:.4f}s')
print(f'Total time   : {train_time + pred_time:.4f}s')

# Release the broadcast variable from executor memory when we are done.
bc_log_prob_table.unpersist()
counts_rdd.unpersist()

# TODO: Expected accuracy on the full car evaluation dataset is ~87.39%
# (Zheng 2014). With the 5-row dummy dataset accuracy is not meaningful —
# replace DATA_PATH with the real file path before interpreting results.